# DistilBERT Final Model Training
This notebook trains a DistilBERT model for sequence classification using a predefined set of optimal hyperparameters. The dataset is split into training (80%), validation (10%), and test (10%) sets.

In [14]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report
)

In [15]:
import wandb
import huggingface_hub

os.environ["WANDB_PROJECT"] = "distilbert_degendered_final"

# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")

wandb.init()

In [16]:
# Model and Hyperparameter Configuration
model_name = "distilbert-base-uncased"
model_cache_path = "../scratch/cache/distilbert_degendered_final"

hyperparameters = {
    "learning_rate": 3.7e-05,
    "num_train_epochs": 10,
    "per_device_train_batch_size": 32,
    "weight_decay": 0.03
}

In [17]:
# Data Preparation (80:10:10 Split)
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# First split: 80% train, 20% temp (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: 10% validation, 10% test from the temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)

In [18]:
# Tokenization Function
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding=False, max_length=512)
    tokens["labels"] = example["label"]
    return tokens

# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_dataset = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_val = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

In [19]:
# Metrics Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    # Get classification report
    report = classification_report(labels, preds, output_dict=True, zero_division=0, target_names=['Female', 'Male'])
    
    # Flatten the report for easy logging
    metrics = {
        'accuracy': report['accuracy'],
        'macro_avg_precision': report['macro avg']['precision'],
        'macro_avg_recall': report['macro avg']['recall'],
        'macro_avg_f1': report['macro avg']['f1-score'],
        'weighted_avg_precision': report['weighted avg']['precision'],
        'weighted_avg_recall': report['weighted avg']['recall'],
        'weighted_avg_f1': report['weighted avg']['f1-score'],
        'female_precision': report['Female']['precision'],
        'female_recall': report['Female']['recall'],
        'female_f1': report['Female']['f1-score'],
        'female_support': report['Female']['support'],
        'male_precision': report['Male']['precision'],
        'male_recall': report['Male']['recall'],
        'male_f1': report['Male']['f1-score'],
        'male_support': report['Male']['support']
    }
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return metrics

In [20]:
# Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Female", 1: "Male"},
    label2id={"Female": 0, "Male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
# Training Arguments
final_model_output_dir = "../scratch/final_distilbert_degendered_model"
training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=hyperparameters["per_device_train_batch_size"],
    num_train_epochs=hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    weight_decay=hyperparameters["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_avg_f1",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_distilbert_degendered_training"
)

In [22]:
# Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val, # Use validation set for in-training evaluation
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_605386/4000249224.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [23]:
# Train the Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Avg Precision,Macro Avg Recall,Macro Avg F1,Weighted Avg Precision,Weighted Avg Recall,Weighted Avg F1,Female Precision,Female Recall,Female F1,Female Support,Male Precision,Male Recall,Male F1,Male Support
1,0.614700,0.623762,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
2,0.627600,0.610550,0.690768,0.345384,0.500000,0.408553,0.477160,0.690768,0.564430,0.000000,0.000000,0.000000,278.000000,0.690768,1.000000,0.817105,621.000000
3,0.606700,0.597472,0.703003,0.667341,0.535679,0.493726,0.682387,0.703003,0.617917,0.627907,0.097122,0.168224,278.000000,0.706776,0.974235,0.819228,621.000000
4,0.494800,0.678594,0.707453,0.666581,0.550820,0.524116,0.684717,0.707453,0.636812,0.619048,0.140288,0.228739,278.000000,0.714115,0.961353,0.819492,621.000000
5,0.362400,0.738500,0.654060,0.604039,0.609527,0.605993,0.664240,0.654060,0.658499,0.446254,0.492806,0.468376,278.000000,0.761824,0.726248,0.743611,621.000000
6,0.217000,0.943795,0.669633,0.592414,0.575102,0.577394,0.645706,0.669633,0.652722,0.452736,0.327338,0.379958,278.000000,0.732092,0.822866,0.774829,621.000000
7,0.133500,1.223193,0.662959,0.596049,0.589146,0.591502,0.652162,0.662959,0.656687,0.448980,0.395683,0.420650,278.000000,0.743119,0.782609,0.762353,621.000000
8,0.124500,1.419250,0.649611,0.577880,0.571537,0.573389,0.636881,0.649611,0.642189,0.423237,0.366906,0.393064,278.000000,0.732523,0.776167,0.753714,621.000000
9,0.067500,1.561574,0.676307,0.595379,0.569999,0.570630,0.645941,0.676307,0.651902,0.462857,0.291367,0.357616,278.000000,0.727901,0.848631,0.783643,621.000000
10,0.042200,1.640582,0.659622,0.578331,0.563882,0.565165,0.634660,0.659622,0.642489,0.430693,0.312950,0.362500,278.000000,0.725968,0.814815,0.767830,621.000000


Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[  0 278]
 [  0 621]]
Confusion Matrix:
 [[ 27 251]
 [ 16 605]]
Confusion Matrix:
 [[ 39 239]
 [ 24 597]]
Confusion Matrix:
 [[137 141]
 [170 451]]
Confusion Matrix:
 [[ 91 187]
 [110 511]]
Confusion Matrix:
 [[110 168]
 [135 486]]
Confusion Matrix:
 [[102 176]
 [139 482]]
Confusion Matrix:
 [[ 81 197]
 [ 94 527]]
Confusion Matrix:
 [[ 87 191]
 [115 506]]


TrainOutput(global_step=2250, training_loss=0.3259108919567532, metrics={'train_runtime': 166.7464, 'train_samples_per_second': 431.134, 'train_steps_per_second': 13.494, 'total_flos': 9523081289379840.0, 'train_loss': 0.3259108919567532, 'epoch': 10.0})

In [24]:
# Final Evaluation on the Test Set
print("--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")

print("\nFinal Test Set Evaluation Results:")
print(test_results)

--- Final Evaluation on Test Set ---


Confusion Matrix:
 [[149 130]
 [208 412]]

Final Test Set Evaluation Results:
{'test_loss': 0.753848671913147, 'test_accuracy': 0.6240266963292548, 'test_macro_avg_precision': 0.5887572741273631, 'test_macro_avg_recall': 0.5992831541218638, 'test_macro_avg_f1': 0.5888378311088018, 'test_weighted_avg_precision': 0.6537673982940931, 'test_weighted_avg_recall': 0.6240266963292548, 'test_weighted_avg_f1': 0.6344629377254347, 'test_female_precision': 0.4173669467787115, 'test_female_recall': 0.5340501792114696, 'test_female_f1': 0.46855345911949686, 'test_female_support': 279.0, 'test_male_precision': 0.7601476014760148, 'test_male_recall': 0.6645161290322581, 'test_male_f1': 0.7091222030981067, 'test_male_support': 620.0, 'test_runtime': 1.3205, 'test_samples_per_second': 680.801, 'test_steps_per_second': 21.961, 'epoch': 10.0}


In [25]:
# Save the Final Model and Tokenizer
trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)
print(f"Final model and tokenizer saved to: {final_model_output_dir}")

Final model and tokenizer saved to: ../scratch/final_distilbert_degendered_model
